## Example 1: Basic mapping implementations

In [ ]:
import zict
import pickle

# Example 1: Basic mapping implementations
print("Basic mapping implementations:")

# LRU cache mapping
lru = zict.LRU(3, {})  # Max 3 items
print(f"LRU cache created with max size 3")

# Add items
lru['a'] = 1
lru['b'] = 2
lru['c'] = 3
lru['d'] = 4  # This should evict 'a'

print(f"Items in LRU: {dict(lru)}")
print(f"LRU size: {len(lru)}")

# Access order affects LRU
_ = lru['b']  # Access 'b' to make it most recent
lru['e'] = 5  # Should evict 'c' now

print(f"After accessing 'b' and adding 'e': {dict(lru)}")

# File-based mapping (requires bytes)
import tempfile
import os

temp_dir = tempfile.mkdtemp()
file_map = zict.File(temp_dir)

# Store bytes (File mapping expects bytes)
file_map['key1'] = b'Hello World'
file_map['key2'] = pickle.dumps({'nested': 'data'})

print(f"File mapping keys: {list(file_map.keys())}")
print(f"Value from key1: {file_map['key1']}")
print(f"Value from key2: {pickle.loads(file_map['key2'])}")
print(f"Files created: {os.listdir(temp_dir)}")

# Cleanup
import shutil
shutil.rmtree(temp_dir)

print("Basic mapping implementations completed")

## Example 2: Chained and layered mappings

In [ ]:
import zict
import tempfile
import os

# Example 2: Chained and layered mappings
print("Chained and layered mappings:")

# Create a layered mapping using Buffer (fast -> slow)
fast = {}  # In-memory dict
slow = {}  # Backup storage

# Buffer combines two mappings with overflow behavior
buffer_chain = zict.Buffer(fast, slow, n=2)  # Keep only 2 items in fast

print(f"Created buffer chain: fast (max 2) -> slow")

# Add items
buffer_chain['item1'] = 'Fast access'
buffer_chain['item2'] = 'Also fast'
print(f"After 2 items - Fast: {fast}, Slow: {slow}")

buffer_chain['item3'] = 'Triggers overflow'
print(f"After 3 items - Fast: {fast}, Slow: {slow}")

# Buffer mapping with LRU eviction
lru_fast = zict.LRU(3, {})
lru_slow = {}
buffer_lru = zict.Buffer(lru_fast, lru_slow, n=3)

print(f"\nBuffer with LRU:")
for i in range(6):
    buffer_lru[f'key_{i}'] = f'value_{i}'

print(f"Fast (LRU): {dict(lru_fast)}")
print(f"Slow: {lru_slow}")

# Zip file-based mapping
temp_zip = tempfile.NamedTemporaryFile(suffix='.zip', delete=False)
temp_zip.close()

zip_map = zict.Zip(temp_zip.name)
zip_map['data1'] = b'First entry'
zip_map['data2'] = b'Second entry'
zip_map.flush()

print(f"\nZip file mapping:")
print(f"Keys in zip: {list(zip_map.keys())}")
print(f"Value for 'data1': {zip_map['data1']}")

zip_map.close()
os.unlink(temp_zip.name)

# Sieve for conditional routing
def is_small(key, value):
    """Selector: returns True for small values, False for large"""
    return len(str(value)) <= 10

small_items = {}
large_items = {}
mappings = {True: small_items, False: large_items}
sieve_map = zict.Sieve(mappings, is_small)

sieve_map['short'] = 'hi'
sieve_map['long'] = 'This is a much longer string'
sieve_map['medium'] = 'medium len'

print(f"\nSieve mapping by value size:")
print(f"Small items (<=10 chars): {small_items}")
print(f"Large items (>10 chars): {large_items}")

print("Chained and layered mappings completed")

## Example 3: Serialization and compression

In [ ]:
import zict
import pickle

# Example 3: Serialization and compression
print("Serialization and compression:")

# Func mapping applies functions on get/set
def serialize(obj):
    return pickle.dumps(obj)

def deserialize(data):
    return pickle.loads(data)

serializing_map = zict.Func(serialize, deserialize, {})

# Store Python objects
serializing_map['list'] = [1, 2, 3, 4, 5]
serializing_map['dict'] = {'key': 'value', 'number': 42}

print(f"Stored objects in serializing map")
print(f"Retrieved list: {serializing_map['list']}")
print(f"Retrieved dict: {serializing_map['dict']}")

# Check raw storage (serialized form)
print(f"Raw storage type: {type(list(serializing_map.d.values())[0])}")

# Compress mapping
try:
    from lz4 import block
    
    def compress(data):
        return block.compress(data)
    
    def decompress(data):
        return block.decompress(data)
    
    # Chain serialization and compression (correct order: serialize THEN compress)
    base_map = {}
    # First layer: compress/decompress bytes
    compressed_layer = zict.Func(compress, decompress, base_map)
    # Second layer: serialize/deserialize objects to bytes
    compressed_map = zict.Func(serialize, deserialize, compressed_layer)
    
    # Store large data
    large_data = list(range(1000))
    compressed_map['large_list'] = large_data
    
    print(f"\nCompressed storage enabled")
    print(f"Retrieved large list length: {len(compressed_map['large_list'])}")
    print(f"Raw compressed size vs original: {len(base_map[list(base_map.keys())[0]])} bytes compressed")
    
except ImportError:
    print("\nlz4 not available, skipping compression example")

print("\nSerialization and compression completed")

## Example 4: Thread-safe and callback mappings

In [ ]:
import zict
import threading
from collections import defaultdict

# Example 4: Callbacks and thread-safe mappings
print("Callbacks and thread-safe mappings:")

# LRU with eviction callback
access_log = []

def on_evict(key, value):
    """Called when items are evicted from LRU"""
    access_log.append(f"Evicted: {key} = {value}")
    print(f"  Evicted key '{key}' (value: {value})")

lru_with_callback = zict.LRU(3, {}, on_evict=on_evict)

print("LRU with eviction callbacks:")
lru_with_callback['a'] = 1
lru_with_callback['b'] = 2
lru_with_callback['c'] = 3
print(f"  Added 3 items, no evictions yet")

lru_with_callback['d'] = 4  # Should evict 'a'
lru_with_callback['e'] = 5  # Should evict 'b'

print(f"Eviction log: {access_log}")

# Buffer with callbacks for data movement
move_log = []

def log_move(key, value):
    """Called when data moves from fast to slow storage"""
    move_log.append(f"{key}: fast->slow")
    print(f"  Moving {key} from fast to slow storage")

fast = {}
slow = {}
buffer_with_callbacks = zict.Buffer(
    fast, slow, n=2, 
    fast_to_slow_callbacks=log_move
)

print(f"\nBuffer with movement callbacks:")
buffer_with_callbacks['item1'] = 'data1'
buffer_with_callbacks['item2'] = 'data2'
buffer_with_callbacks['item3'] = 'data3'  # Triggers movement

print(f"Movement log: {move_log}")

# Func for value transformation with side effects
call_count = {'get': 0, 'set': 0}

def transform_set(value):
    call_count['set'] += 1
    return value.upper() if isinstance(value, str) else value

def transform_get(value):
    call_count['get'] += 1
    return value.lower() if isinstance(value, str) else value

func_map = zict.Func(transform_set, transform_get, {})
func_map['greeting'] = 'Hello World'
func_map['message'] = 'Testing Zict'

print(f"\nFunc with transformation tracking:")
print(f"  Stored (uppercase): {dict(func_map.d)}")
print(f"  Retrieved 'greeting': {func_map['greeting']}")
print(f"  Retrieved 'message': {func_map['message']}")
print(f"  Call count: {call_count}")

print("\nCallbacks and thread-safe mappings completed")

## Example 5: Production caching patterns

In [ ]:
import zict
import tempfile
import time
import os
import pickle

# Example 5: Production caching patterns
print("Production caching patterns:")

class ProductionCache:
    """Multi-level cache with memory, disk, and eviction policies"""
    
    def __init__(self, memory_size=100, disk_dir=None):
        # Level 1: In-memory LRU cache (fastest)
        self.memory = zict.LRU(memory_size, {})
        
        # Level 2: Disk storage with serialization (File expects bytes)
        if disk_dir is None:
            disk_dir = tempfile.mkdtemp()
        
        # Wrap File with Func for automatic pickle serialization
        file_store = zict.File(disk_dir)
        self.disk = zict.Func(pickle.dumps, pickle.loads, file_store)
        self.disk_dir = disk_dir
        
        # Combine with buffer for write optimization
        self.cache = zict.Buffer(
            self.memory,     # Fast storage
            self.disk,       # Slow storage  
            n=10             # Buffer 10 items before flushing
        )
        
        self.stats = {'hits': 0, 'misses': 0, 'sets': 0}
    
    def get(self, key):
        try:
            value = self.cache[key]
            self.stats['hits'] += 1
            return value
        except KeyError:
            self.stats['misses'] += 1
            raise
    
    def set(self, key, value):
        self.cache[key] = value
        self.stats['sets'] += 1
    
    def get_stats(self):
        total_requests = self.stats['hits'] + self.stats['misses']
        hit_rate = self.stats['hits'] / total_requests if total_requests > 0 else 0
        return {
            'hit_rate': f"{hit_rate:.2%}",
            'total_requests': total_requests,
            'memory_items': len(self.memory),
            **self.stats
        }

# Create production cache
cache = ProductionCache(memory_size=5)

print("Testing production cache:")

# Simulate workload
for i in range(20):
    key = f"item_{i}"
    cache.set(key, f"Data for {key} - {time.time()}")

# Access some items multiple times (should hit memory)
for i in range(5):
    try:
        data = cache.get(f"item_{i}")
        print(f"Retrieved: item_{i}")
    except KeyError:
        print(f"Cache miss: item_{i}")

# Access older items (should hit disk)
for i in range(15, 20):
    try:
        data = cache.get(f"item_{i}")
        print(f"Retrieved from disk: item_{i}")
    except KeyError:
        print(f"Cache miss: item_{i}")

# Show cache statistics
stats = cache.get_stats()
print(f"\nCache performance:")
for key, value in stats.items():
    print(f"  {key}: {value}")

# Advanced: Sieve for routing by data size
print(f"\nSize-aware routing with Sieve:")

def is_small_data(key, value):
    """Route small data to memory, large to disk"""
    return len(str(value)) < 100

small_storage = {}
large_storage = {}
mappings = {True: small_storage, False: large_storage}
size_aware_cache = zict.Sieve(mappings, is_small_data)

size_aware_cache['small'] = "tiny"
size_aware_cache['large'] = "x" * 200

print(f"Small storage: {list(small_storage.keys())}")
print(f"Large storage: {list(large_storage.keys())}")
print(f"Total items: {len(size_aware_cache)}")

# Cleanup
import shutil
shutil.rmtree(cache.disk_dir, ignore_errors=True)

print("\nProduction caching patterns completed")